# Preliminary Knowledge Notebook: Narrowband, ULA, and MUSIC

This notebook summarizes key concepts from `docs/preliminary_knowledge.md` and turns them into runnable mini-experiments.

Audience: self-study for array signal processing and DOA estimation.

Main thread:
1. Why narrowband is a relative condition (`B * Delta_tau << 1`)
2. How ULA steering vectors come from geometry
3. How MUSIC uses the noise subspace orthogonality
4. Why wideband mismatch makes peaks lower/wider


## Setup

We use only `numpy` and deterministic random seeds for reproducibility.


In [1]:
from __future__ import annotations

import numpy as np

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(7)

c = 3e8
deg = np.pi / 180

print('Environment ready.')


Environment ready.


## 1) Narrowband Criterion as a Computable Quantity

For a signal bandwidth `B` and array aperture delay `Delta_tau`, a common engineering condition is:

`B * Delta_tau << 1`

- `Delta_tau` comes from geometry (array size and angle), not directly from carrier frequency
- if this product is small, delay is approximated well by phase rotation


In [2]:
def max_aperture_delay(D: float, c0: float = c) -> float:
    """Worst-case delay across aperture D (meters)."""
    return D / c0

B = 20e6  # 20 MHz
for D in [1.0, 100.0]:
    dt = max_aperture_delay(D)
    metric = B * dt
    print(f'D={D:>6.1f} m, Delta_tau={dt*1e9:>8.3f} ns, B*Delta_tau={metric:>7.4f}')


D=   1.0 m, Delta_tau=   3.333 ns, B*Delta_tau= 0.0667
D= 100.0 m, Delta_tau= 333.333 ns, B*Delta_tau= 6.6667


## 2) ULA Steering Vector from Geometry

For a ULA with `N` sensors and spacing `d`, under a plane wave with angle `theta`:

`a(theta)[n] = exp(-j * 2*pi/lambda * n * d * sin(theta))`

(using zero-based index `n=0,...,N-1`).

We test the common case `d=lambda/2`, `N=4`, `theta=30 deg`.


In [3]:
def steering_ula(theta_rad: float, N: int, d: float, wavelength: float) -> np.ndarray:
    n = np.arange(N)
    phase = -2.0 * np.pi / wavelength * n * d * np.sin(theta_rad)
    return np.exp(1j * phase)

N = 4
theta = 30 * deg
wavelength = 1.0
d = wavelength / 2
a = steering_ula(theta, N, d, wavelength)

print('steering vector:', a)
print('phase (rad):', np.angle(a))


steering vector: [ 1.+0.j  0.-1.j -1.-0.j -0.+1.j]
phase (rad): [ 0.     -1.5708 -3.1416  1.5708]


## 3) MUSIC Core: Signal/Noise Subspace Separation

Narrowband snapshot model:

`x[k] = A s[k] + n[k]`

For one source (`p=1`) and white noise, EVD of sample covariance should show:
- one dominant eigenvalue
- remaining eigenvalues near noise floor


In [4]:
def simulate_snapshots(theta0_deg: float, N: int = 8, K: int = 600, snr_db: float = 5.0):
    theta0 = theta0_deg * deg
    a0 = steering_ula(theta0, N=N, d=0.5, wavelength=1.0)[:, None]

    s = (rng.standard_normal((1, K)) + 1j * rng.standard_normal((1, K))) / np.sqrt(2)
    signal = a0 @ s

    sig_power = np.mean(np.abs(signal) ** 2)
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.sqrt(noise_power / 2) * (
        rng.standard_normal((N, K)) + 1j * rng.standard_normal((N, K))
    )

    x = signal + noise
    R = (x @ x.conj().T) / K
    return R

R = simulate_snapshots(theta0_deg=20, N=8, K=800, snr_db=0.0)
eigs = np.linalg.eigvalsh(R)[::-1]
print('Eigenvalues (descending):')
print(eigs)


Eigenvalues (descending):
[8.9213 1.0844 1.0696 1.0143 0.9838 0.911  0.8774 0.8271]


/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: divide by zero encountered in matmul
  R = (x @ x.conj().T) / K
/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: overflow encountered in matmul
  R = (x @ x.conj().T) / K
/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: invalid value encountered in matmul
  R = (x @ x.conj().T) / K


## 4) MUSIC Pseudospectrum and Peak Detection

MUSIC spectrum:

`P(theta) = 1 / (a(theta)^H Un Un^H a(theta))`

A sharp peak is expected near the true DOA.


In [5]:
def music_spectrum(R: np.ndarray, p: int, theta_grid_deg: np.ndarray, N: int) -> np.ndarray:
    vals, vecs = np.linalg.eigh(R)
    order = np.argsort(vals)[::-1]
    vecs = vecs[:, order]
    Un = vecs[:, p:]

    spec = np.zeros_like(theta_grid_deg, dtype=float)
    for i, th in enumerate(theta_grid_deg):
        a = steering_ula(th * deg, N=N, d=0.5, wavelength=1.0)[:, None]
        denom = (a.conj().T @ Un @ Un.conj().T @ a).real.item()
        spec[i] = 1.0 / max(denom, 1e-12)
    return spec

N = 8
R = simulate_snapshots(theta0_deg=20, N=N, K=1000, snr_db=2.0)
grid = np.linspace(-60, 60, 721)
P = music_spectrum(R, p=1, theta_grid_deg=grid, N=N)
theta_hat = grid[np.argmax(P)]
print(f'Estimated DOA from MUSIC: {theta_hat:.2f} deg')


Estimated DOA from MUSIC: 20.00 deg


/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: divide by zero encountered in matmul
  R = (x @ x.conj().T) / K
/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: overflow encountered in matmul
  R = (x @ x.conj().T) / K
/var/folders/83/j6l7_w4x2bdg4ms_fzfwcv3h0000gn/T/ipykernel_91981/1664904136.py:15: RuntimeWarning: invalid value encountered in matmul
  R = (x @ x.conj().T) / K


## 5) Wideband Mismatch Demo (Single Narrowband Steering Used Everywhere)

When wideband data is forced into a single center-frequency steering model, different frequencies are not perfectly aligned.

Common symptom: the spatial peak gets lower/wider and may shift.

Below, we compare:
- ideal narrowband response at `f_c`
- averaged response from multiple sub-frequencies but still evaluated with center-frequency model


In [6]:
def matched_gain(theta_scan_deg: np.ndarray, theta_true_deg: float, N: int, f: float, d: float, c0: float = c):
    """Reference gain when both steering and signal use same frequency f."""
    lam = c0 / f
    a_true = steering_ula(theta_true_deg * deg, N=N, d=d, wavelength=lam)[:, None]
    gain = np.zeros_like(theta_scan_deg, dtype=float)
    for i, th in enumerate(theta_scan_deg):
        a_scan = steering_ula(th * deg, N=N, d=d, wavelength=lam)[:, None]
        gain[i] = np.abs((a_scan.conj().T @ a_true).item()) ** 2
    return gain / np.max(gain)

def mismatched_gain(theta_scan_deg: np.ndarray, theta_true_deg: float, N: int, f_true: float, f_steer: float, d: float, c0: float = c):
    """Beamformer is fixed at f_steer while signal actually comes at f_true."""
    lam_true = c0 / f_true
    lam_steer = c0 / f_steer
    a_true = steering_ula(theta_true_deg * deg, N=N, d=d, wavelength=lam_true)[:, None]
    gain = np.zeros_like(theta_scan_deg, dtype=float)
    for i, th in enumerate(theta_scan_deg):
        a_scan = steering_ula(th * deg, N=N, d=d, wavelength=lam_steer)[:, None]
        gain[i] = np.abs((a_scan.conj().T @ a_true).item()) ** 2
    return gain

N = 16
fc = 10e9
B = 2e9
d = (c / fc) / 2
theta_true = 25.0
scan = np.linspace(-60, 60, 1201)

g_nb = matched_gain(scan, theta_true, N, f=fc, d=d)

sub_freqs = np.linspace(fc - B / 2, fc + B / 2, 31)
g_wb = np.zeros_like(scan)
for f in sub_freqs:
    g_wb += mismatched_gain(scan, theta_true, N, f_true=f, f_steer=fc, d=d)
g_wb /= np.max(g_wb)

peak_nb = scan[np.argmax(g_nb)]
peak_wb = scan[np.argmax(g_wb)]

thr = 0.5
bw_nb = scan[g_nb >= thr]
bw_wb = scan[g_wb >= thr]
width_nb = bw_nb[-1] - bw_nb[0]
width_wb = bw_wb[-1] - bw_wb[0]

print(f'Narrowband peak at {peak_nb:.2f} deg, width@-3dB ~ {width_nb:.2f} deg')
print(f'Wideband mismatched peak at {peak_wb:.2f} deg, width@-3dB ~ {width_wb:.2f} deg')


Narrowband peak at 25.00 deg, width@-3dB ~ 6.90 deg
Wideband mismatched peak at 25.00 deg, width@-3dB ~ 7.70 deg


## Summary and Self-Check

Key takeaways:
1. `B * Delta_tau` is the compact test for narrowband validity
2. ULA steering vectors encode geometric delay as linear phase progression
3. MUSIC peak comes from steering/noise-subspace near-orthogonality
4. Wideband mismatch usually broadens/weakens peaks before creating many false peaks

Suggested exercises:
- Change `N`, `K`, and `snr_db` and inspect eigenvalue separation
- Try two close sources and compare MUSIC peak resolvability
- Increase `B` in the wideband demo and track peak broadening/shift
